In [ ]:
import pandas as pd
from pathlib import Path
path = Path("..") / "data" / "MosquitoDat.xlsx"  # see data/README.md
mosquito_data = pd.read_excel(path, engine="openpyxl")
mosquito_data.head()  
#test to see if notebook can access file

In [ ]:
xls = pd.ExcelFile(path, engine="openpyxl")
print(xls.sheet_names)    
#identifies both sheets in excel file

In [ ]:
all_xls =pd.read_excel(path, 
    sheet_name=None, 
    engine="openpyxl")  
#reads every sheet into a dictionary

for name, df in all_xls.items():
    print(f"\n Sheet: {name} ")
    display(df)  
#illustrates both tables

In [ ]:
sheets = pd.read_excel(
    path,
    sheet_name=['Trial1', 'Trial2', 'Key1', 'Key2'], 
    header=0,                          
    engine="openpyxl")  
#reads sheets 

Data1_df = sheets ['Trial1']
Key1_df = sheets ['Key1'] 
Data2_df = sheets ['Trial2']
Key2_df = sheets ['Key2']  
#turns tables into dataframes to use

print(Data1_df)
print(Key1_df)  
print(Data2_df)
print(Key2_df) 

In [ ]:
print(Data1_df.columns.tolist())
print(Key1_df.columns.tolist())
print(Data2_df.columns.tolist())
print(Key2_df.columns.tolist()) 
#list column names 

In [ ]:
merged1_df = Data1_df.merge(
    Key1_df[['Blind ID','Genotype','Density']],
    on='Blind ID',
    how='left'
).drop(columns=['Blind ID'])

merged2_df = Data2_df.merge(
    Key2_df[['Blind ID','Genotype','Density']],
    on='Blind ID',
    how='left'
).drop(columns=['Blind ID']) 

#merges the Blind ID to the proper genotype and density 

print(merged1_df)
print(merged2_df)

In [ ]:
merged1_df = (
    merged1_df
      .rename(columns={'Density_x': 'Density'}) 
      .drop(columns=['Density_y']))  

merged2_df = (
    merged2_df
      .rename(columns={'Density_x': 'Density'})  
      .drop(columns=['Density_y']))  
#renames the Density column to provide more clarity (both density columns were necessary for merging to ensure the Genotype is associated to the right Density and Blind ID).

print(merged1_df)
print(merged2_df) 

In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns 
import numpy as np 
#load libraries

In [ ]:
sns.jointplot(data=merged1_df,
              x='Trial Day',
              y='Larvae Count',
              hue='Genotype')  

sns.jointplot(data=merged2_df,
              x='Trial Day',
              y='Larvae Count',
              hue='Genotype') 

#experimenting with a jointplot of larvae over time 

In [ ]:
def clean_data(df):
    
    all_days = df  # unfiltered data, used for mortality
    df = df[df['Trial Day'] >= 3].copy() 
    #keeps trial days 3 and later (days 1-2 are excluded from every calculation below)

    avg_pupation = (
        df[df['Pupae Emerged'] > 0]
        .groupby(['Genotype', 'Density'])['Trial Day']
        .mean()
        .unstack()) 
    #mean trial day among days when pupae were recorded (not weighted by how many pupae emerged)

    avg_emergence = (
        df[df['Mosquitoes Emerged'] > 0]
        .groupby(['Genotype', 'Density'])['Trial Day']
        .mean()
        .unstack()) 
    #mean trial day among days when adults were recorded (not weighted by how many emerged)

    total_dead     = all_days.groupby(['Genotype', 'Density'])['# of Deceased'].sum()
    start_larvae   = all_days[all_days['Trial Day'] == 1].groupby(['Genotype', 'Density'])['Larvae Count'].sum()
    avg_mortality  = (total_dead / start_larvae).unstack()
    #mortality = deaths on all days / larvae at the start, summed over all replicate containers in each genotype x density group

    total_males    = df.groupby(['Genotype', 'Density'])['Males Emerged'].sum()
    total_females  = df.groupby(['Genotype', 'Density'])['Females Emerged'].sum()
    avg_sex_ratio  = (total_males / total_females).unstack() 
    # calculates the sex ratio by finding the sum of males and females individually and dividing males by females

    return avg_pupation, avg_emergence, avg_mortality, avg_sex_ratio

In [ ]:
for df, label in [(merged1_df, 'Trial 1'), (merged2_df, 'Trial 2')]:  
    avg_pupation, avg_emergence, avg_mortality, avg_sex_ratio = clean_data(df)  
    #creates a loop to run the following graph codes through trial 1 and trial 2 

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    avg_pupation.plot.bar(ax=axes[0,0], rot=1)
    axes[0,0].set(
        title=f'{label} – Avg. Pupation Day',
        ylabel='Day') 
    #bar graph for average pupation in the top left

    avg_emergence.plot.bar(ax=axes[0,1], rot=1)
    axes[0,1].set(
        title=f'{label} – Avg. Emergence Day',
        ylabel='Day') 
    #bar graph for average mosquito emergence (no gender included) in top right

    avg_mortality.plot.bar(ax=axes[1,0], rot=1)
    axes[1,0].set(
        title=f'{label} – Mortality Rate',
        ylabel='Fraction Deceased') 
    #bar graph for mortality rate in bottom left

    avg_sex_ratio.plot.bar(ax=axes[1,1], rot=1)
    axes[1,1].set(
        title=f'{label} – Sex Ratio (M:F)',
        ylabel='Ratio') 
    #bar graph for sex ratio M to F in bottom right

    plt.tight_layout()
    plt.show()
    #load relevant graphs to conclude the fitness cost of carrying the pesticide resistant genotypes at high(100) and low(25) densities